# AutoResearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：**
1. Runtime → Change runtime type → 選 **GPU (T4+)**
2. 從上到下依序執行每個 cell
3. 訓練完成後結果會自動回傳到 MD.Piece 後端

**功能：**
- 單次訓練 baseline 或自訂實驗
- 自動實驗循環（多次迭代）
- 結果自動回傳 + results.tsv 批次匯入
- T4 自動偵測與 batch size 調整

In [ ]:
# Step 1: 確認 GPU 並偵測型號
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → GPU")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_mem / 1e9
GPU_COMPUTE = torch.cuda.get_device_capability(0)
print(f"GPU: {GPU_NAME}")
print(f"VRAM: {GPU_VRAM_GB:.1f} GB")
print(f"Compute Capability: {GPU_COMPUTE[0]}.{GPU_COMPUTE[1]}")

# 判斷 GPU 等級
IS_T4 = 'T4' in GPU_NAME
IS_A100 = 'A100' in GPU_NAME
IS_H100 = 'H100' in GPU_NAME
IS_LOW_VRAM = GPU_VRAM_GB < 20

if IS_T4:
    print("\n⚠️  T4 偵測到 — 將自動調低 batch size 以避免 OOM")
    print("   預期訓練時間：~15-20 分鐘（vs H100 ~5 分鐘）")
elif IS_A100:
    print("\n✅ A100 — 效能良好")
elif IS_H100:
    print("\n✅ H100 — 最佳效能，支援 Flash Attention 3")
else:
    print(f"\nℹ️  未知 GPU: {GPU_NAME}，將嘗試以預設設定執行")

In [ ]:
# Step 2: 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
# Step 3: Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

In [ ]:
# Step 4: 安裝依賴
!uv sync

In [ ]:
# Step 5: T4 安全措施 — 自動調整 train.py 參數
import re

def patch_train_py_for_gpu():
    """根據 GPU VRAM 自動調整 batch size 等參數"""
    with open('train.py', 'r') as f:
        code = f.read()
    original = code

    if IS_LOW_VRAM:
        # T4 (15GB) 需要更小的 batch size
        patches = {
            'TOTAL_BATCH_SIZE': 2**17,   # 131072 (vs default 2**19)
        }
        for param, val in patches.items():
            pattern = re.compile(rf'^({param}\s*=\s*)(.+)$', re.MULTILINE)
            if pattern.search(code):
                old_val = pattern.search(code).group(2)
                code = pattern.sub(rf'\g<1>{val}', code)
                print(f"  調整 {param}: {old_val} → {val}")

    if code != original:
        with open('train.py', 'w') as f:
            f.write(code)
        print(f"\n已自動調整 train.py 以適應 {GPU_NAME} ({GPU_VRAM_GB:.0f}GB)")
    else:
        print(f"GPU {GPU_NAME} 無需調整參數")

patch_train_py_for_gpu()

In [ ]:
# Step 6: 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

In [ ]:
# Step 7: 設定 MD.Piece 後端連線
# ⚠️ 如果後端跑在本機，請用 ngrok 或改成你的公開 URL
MDPIECE_API = "http://localhost:8000"  # ← 改成你的後端 URL

import requests
import json

def check_mdpiece_connection():
    """測試 MD.Piece API 連線"""
    try:
        resp = requests.get(f"{MDPIECE_API}/", timeout=5)
        data = resp.json()
        print(f"✅ 已連線到 MD.Piece API: {data.get('message', '')}")
        return True
    except requests.exceptions.ConnectionError:
        print(f"❌ 無法連線到 {MDPIECE_API}")
        print("   請確認：")
        print("   1. 後端是否已啟動 (uvicorn backend.main:app --port 8000)")
        print("   2. 若在 Colab 使用，需要 ngrok 或公開 URL")
        print("   3. 可先跳過此步驟，訓練完後手動匯入結果")
        return False
    except Exception as e:
        print(f"⚠️  連線異常: {e}")
        return False

API_CONNECTED = check_mdpiece_connection()

In [ ]:
# Step 8: 訓練（T4 約 15-20 分鐘，A100/H100 約 5 分鐘）
import time
import subprocess
import re

def run_experiment(name="baseline", retry_on_oom=True):
    """執行一次訓練並回傳結果 dict，OOM 時自動重試"""
    start = time.time()
    result = subprocess.run(
        ["uv", "run", "train.py"],
        capture_output=True, text=True
    )
    duration = time.time() - start

    # OOM 偵測與自動重試
    if result.returncode != 0 and 'out of memory' in (result.stderr + result.stdout).lower():
        print("⚠️  OOM 偵測到！自動調低 TOTAL_BATCH_SIZE 並重試...")
        if retry_on_oom:
            with open('train.py', 'r') as f:
                code = f.read()
            pattern = re.compile(r'^(TOTAL_BATCH_SIZE\s*=\s*)(\d+)', re.MULTILINE)
            match = pattern.search(code)
            if match:
                old_val = int(match.group(2))
                new_val = old_val // 2
                code = pattern.sub(rf'\g<1>{new_val}', code)
                with open('train.py', 'w') as f:
                    f.write(code)
                print(f"   TOTAL_BATCH_SIZE: {old_val} → {new_val}")
                torch.cuda.empty_cache()
                return run_experiment(name, retry_on_oom=False)

    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[-500:])

    val_bpb = None
    train_loss = None
    steps = None

    for line in result.stdout.split("\n"):
        m = re.search(r'val_bpb[=:\s]+(\d+\.\d+)', line)
        if m:
            val_bpb = float(m.group(1))
        m = re.search(r'(?:train_)?loss[=:\s]+(\d+\.\d+)', line)
        if m:
            train_loss = float(m.group(1))
        m = re.search(r'step[=:\s]+(\d+)', line)
        if m:
            steps = int(m.group(1))

    print(f"\n--- 結果 ---")
    print(f"val_bpb: {val_bpb}")
    print(f"train_loss: {train_loss}")
    print(f"steps: {steps}")
    print(f"duration: {duration:.0f}s ({duration/60:.1f}m)")
    if result.returncode != 0:
        print(f"⚠️  非零退出碼: {result.returncode}")

    return {
        "name": name,
        "val_bpb": val_bpb,
        "train_loss": train_loss,
        "steps": steps,
        "duration_seconds": duration,
        "returncode": result.returncode,
    }

# 執行 baseline
baseline = run_experiment("colab-baseline")

In [ ]:
# Step 9: 回傳結果到 MD.Piece 後端

def submit_to_mdpiece(exp_result, kept=None, notes_extra=""):
    """提交實驗結果到 MD.Piece API"""
    payload = {
        "name": exp_result["name"],
        "val_bpb": exp_result["val_bpb"],
        "train_loss": exp_result["train_loss"],
        "steps": exp_result["steps"],
        "duration_seconds": exp_result["duration_seconds"],
        "notes": f"Colab {GPU_NAME} ({GPU_VRAM_GB:.0f}GB), PyTorch {torch.__version__}" + (f" | {notes_extra}" if notes_extra else ""),
        "model_config_summary": "default autoresearch config" + (" (T4-adjusted)" if IS_LOW_VRAM else ""),
        "kept": kept,
    }

    if not API_CONNECTED:
        print(f"⚠️  API 未連線，儲存結果供手動匯入：")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return False

    try:
        resp = requests.post(f"{MDPIECE_API}/research/", json=payload, timeout=10)
        resp.raise_for_status()
        print(f"✅ 結果已回傳: {resp.json().get('message', '')}")
        return True
    except Exception as e:
        print(f"❌ 無法回傳到 MD.Piece 後端: {e}")
        print(f"手動提交資料:\n{json.dumps(payload, indent=2, ensure_ascii=False)}")
        return False

submit_to_mdpiece(baseline, kept=True)

In [ ]:
# Step 10（選用）: 自動實驗循環
# 設定要跑幾輪實驗
NUM_ITERATIONS = 3  # ← 改成你要的輪數

import datetime
import csv
import random

best_bpb = baseline["val_bpb"]
results = [baseline]

# 超參數搜尋空間（根據 GPU 等級調整範圍）
if IS_LOW_VRAM:
    HYPERPARAM_VARIANTS = [
        ("DEPTH", [4, 6, 8, 10]),
        ("ASPECT_RATIO", [32, 48, 64]),
        ("TOTAL_BATCH_SIZE", [2**16, 2**17, 2**18]),
        ("LR", [0.0004, 0.0006, 0.0008, 0.001]),
        ("MUON_LR", [0.01, 0.015, 0.02, 0.025]),
        ("WARMUP_STEPS", [0, 50, 100, 150]),
        ("WARMDOWN_STEPS", [200, 300, 500, 700]),
    ]
else:
    HYPERPARAM_VARIANTS = [
        ("DEPTH", [6, 8, 10, 12, 14]),
        ("ASPECT_RATIO", [48, 64, 80, 96]),
        ("TOTAL_BATCH_SIZE", [2**18, 2**19, 2**20]),
        ("LR", [0.0004, 0.0006, 0.0008, 0.001, 0.0015]),
        ("MUON_LR", [0.01, 0.015, 0.02, 0.025, 0.03]),
        ("WARMUP_STEPS", [0, 50, 100, 150, 200]),
        ("WARMDOWN_STEPS", [200, 300, 500, 700, 1000]),
    ]

for i in range(NUM_ITERATIONS):
    # 隨機選一個超參數修改
    param_name, values = random.choice(HYPERPARAM_VARIANTS)
    new_val = random.choice(values)
    exp_name = f"exp-{i+1:03d}-{param_name}={new_val}"

    print(f"\n{'='*60}")
    print(f"實驗 {i+1}/{NUM_ITERATIONS}: {exp_name}")
    print(f"{'='*60}")

    # 讀取 train.py 並修改超參數
    with open("train.py", "r") as f:
        code = f.read()
    original_code = code

    pattern = re.compile(rf'^({param_name}\s*=\s*)(.+)$', re.MULTILINE)
    if pattern.search(code):
        code = pattern.sub(rf'\g<1>{new_val}', code)
        with open("train.py", "w") as f:
            f.write(code)
        print(f"修改 {param_name} = {new_val}")
    else:
        print(f"找不到 {param_name}，跳過")
        continue

    # 清理 GPU 記憶體
    torch.cuda.empty_cache()

    # 執行訓練
    exp = run_experiment(exp_name)

    # 評估結果
    kept = False
    if exp["returncode"] == 0 and exp["val_bpb"] is not None:
        if best_bpb is None or exp["val_bpb"] < best_bpb:
            print(f"✅ 改善！{best_bpb} → {exp['val_bpb']}")
            best_bpb = exp["val_bpb"]
            kept = True
        else:
            print(f"❌ 未改善（{exp['val_bpb']} >= {best_bpb}），還原")
            with open("train.py", "w") as f:
                f.write(original_code)
    else:
        print("⚠️  訓練失敗或無結果，還原")
        with open("train.py", "w") as f:
            f.write(original_code)

    exp["kept"] = kept
    results.append(exp)

    # 回傳到 MD.Piece
    submit_to_mdpiece(exp, kept=kept, notes_extra=f"{param_name}={new_val}")

# 寫入 results.tsv
with open("results.tsv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "val_bpb", "train_loss", "steps", "duration", "kept", "timestamp"], delimiter="\t")
    writer.writeheader()
    for r in results:
        writer.writerow({
            "name": r["name"],
            "val_bpb": r.get("val_bpb", ""),
            "train_loss": r.get("train_loss", ""),
            "steps": r.get("steps", ""),
            "duration": round(r.get("duration_seconds", 0)),
            "kept": r.get("kept", ""),
            "timestamp": datetime.datetime.utcnow().isoformat(),
        })

print(f"\n{'='*60}")
print(f"實驗完成！最佳 val_bpb: {best_bpb}")
print(f"結果已存入 results.tsv（{len(results)} 筆）")
print(f"GPU: {GPU_NAME} ({GPU_VRAM_GB:.0f}GB)")
print(f"{'='*60}")

In [ ]:
# Step 11（選用）: 批次匯入 results.tsv 到 MD.Piece
import os
if os.path.exists("results.tsv"):
    if API_CONNECTED:
        try:
            with open("results.tsv", "rb") as f:
                resp = requests.post(
                    f"{MDPIECE_API}/research/batch",
                    files={"file": ("results.tsv", f, "text/tab-separated-values")},
                    timeout=30,
                )
            data = resp.json()
            print(f"✅ 批次匯入: {data.get('count', 0)} 筆成功")
            if data.get('errors'):
                print(f"⚠️  {data['errors']} 筆失敗")
        except Exception as e:
            print(f"❌ 批次匯入失敗: {e}")
            print("請手動下載 results.tsv 並在 MD.Piece 前端匯入")
    else:
        print("⚠️  API 未連線。請下載 results.tsv 後在 MD.Piece 前端匯入")
        from google.colab import files
        files.download('results.tsv')
else:
    print("results.tsv 不存在，請先執行 Step 10")